# **O pulo do Gato: Modelos especializados**




# Data

In [1]:
!pip install -U datasets huggingface_hub

from datasets import load_dataset

# Load our data
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
data

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [2]:
data["train"][0, -1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

# **Classificação**

* **Classificação de Texto com Modelos Representativos**:
> * Modelos específicos para a tarefa
> * Modelos de incorporação (embedding)
* **Classificação de Texto com Modelos Generativos**:
> * Modelos de linguagem generativos, que podem ser usados para classificação e outras tarefas.


<br>

![imagem](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/fig4-2.png?raw=true)

## **Classificação: Modelo treinado para tarefa**

<br>

![imagem](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/fig4-4.png?raw=true)

In [3]:
import pandas as pd

df_train = pd.DataFrame(data["train"])
df_test = pd.DataFrame(data["test"])

print("Treino (shape):", df_train.shape)
print("Teste (shape):", df_test.shape)

Treino (shape): (8530, 2)
Teste (shape): (1066, 2)


In [4]:
df_train.head()

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1


In [5]:
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm import tqdm
import numpy as np
from sklearn.metrics import classification_report

# Carrega o pipeline correto com o modelo CardiffNLP
pipe_roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device="cuda:0"
)

# Executa a inferência em df_test
y_pred_roberta = []
for output in tqdm(pipe_roberta(KeyDataset(data["test"], "text")), total=len(df_test)):
    # O modelo retorna 'positive', 'neutral' ou 'negative'
    label = output["label"].lower()
    # Mapeando os rótulos para binário (0 = negativo, 1 = positivo/neutro)
    y_pred_roberta.append(0 if label == "negative" else 1)

# Função de avaliação
def evaluate_performance(y_true, y_pred):
    performance = classification_report(
        y_true, y_pred,
        target_names=["Negative Review", "Positive Review"]
    )
    print(performance)

# Avalia o desempenho
evaluate_performance(df_test["label"], y_pred_roberta)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

... (2 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]



  0%|          | 0/1066 [00:00<?, ?it/s]

  0%|          | 1/1066 [00:01<21:52,  1.23s/it]

  0%|          | 2/1066 [00:01<11:08,  1.59it/s]

  1%|          | 7/1066 [00:01<02:25,  7.26it/s]

  1%|          | 11/1066 [00:01<01:31, 11.57it/s]

  1%|▏         | 15/1066 [00:01<01:05, 16.02it/s]

  2%|▏         | 19/1066 [00:01<00:51, 20.47it/s]

  2%|▏         | 23/1066 [00:01<00:43, 23.77it/s]

  3%|▎         | 28/1066 [00:02<00:36, 28.78it/s]

  3%|▎         | 32/1066 [00:02<00:33, 31.02it/s]

  3%|▎         | 36/1066 [00:02<00:32, 31.79it/s]

  4%|▍         | 40/1066 [00:02<00:32, 31.19it/s]

  4%|▍         | 45/1066 [00:02<00:28, 35.76it/s]

  5%|▍         | 50/1066 [00:02<00:26, 38.61it/s]

  5%|▌         | 55/1066 [00:02<00:24, 41.45it/s]

  6%|▌         | 60/1066 [00:02<00:24, 40.88it/s]

  6%|▌         | 65/1066 [00:03<00:24, 41.05it/s]

  7%|▋         | 70/1066 [00:03<00:24, 41.26it/s]

  7%|▋         | 75/1066 [00:03<00:25, 38.89it/s]

  8%|▊         | 81/1066 [00:03<00:23, 41

                 precision    recall  f1-score   support

Negative Review       0.81      0.69      0.75       533
Positive Review       0.73      0.84      0.78       533

       accuracy                           0.77      1066
      macro avg       0.77      0.77      0.77      1066
   weighted avg       0.77      0.77      0.77      1066



## **Classificação: Empregando representações vetoriais (embeddings)**

### Modelos de Aprendizado Supervisionado

In [6]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to embeddings
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [7]:
train_embeddings.shape

(8530, 768)

In [8]:
from sklearn.linear_model import LogisticRegression

# Train a Logistic Regression on our train embeddings
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

LogisticRegression(random_state=42)

In [9]:
# Predict previously unseen instances
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.85      0.86      0.85       533
Positive Review       0.86      0.85      0.85       533

       accuracy                           0.85      1066
      macro avg       0.85      0.85      0.85      1066
   weighted avg       0.85      0.85      0.85      1066



#### Sem usar um classificador?

Podemos aqui calcular a média dos embeddings por classe e aplicar a similaridade de cosseno para prever quais classes melhor correspondem aos documentos:

In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

# Average the embeddings of all documents in each target label
df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
averaged_target_embeddings = df.groupby(768).mean().values

# Find the best matching embeddings between evaluation documents and target embeddings
sim_matrix = cosine_similarity(test_embeddings, averaged_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

# Evaluate the model
evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.85      0.84      0.84       533
Positive Review       0.84      0.85      0.84       533

       accuracy                           0.84      1066
      macro avg       0.84      0.84      0.84      1066
   weighted avg       0.84      0.84      0.84      1066



### Zero-shot Classification

<br>

![imagem](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/fig4-14.png?raw=true)

In [11]:
# Create embeddings for our labels
label_embeddings = model.encode(["A negative review",  "A positive review"])

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

# Find the best matching label for each document
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [13]:
evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.78      0.77      0.78       533
Positive Review       0.77      0.79      0.78       533

       accuracy                           0.78      1066
      macro avg       0.78      0.78      0.78      1066
   weighted avg       0.78      0.78      0.78      1066



## **Classificação: Modelos Generativos + `prompt`**

<br>

![imagem](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/fig4-18.png?raw=true)

### Encoder-decoder Models

In [18]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Carrega o modelo e o tokenizador explicitamente
model_name = "google/flan-t5-small"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda:0")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Criada uma função substituta idêntica ao comportamento do pipeline textgeneration que foi removido... :-(... para manter a compatibilidade com o resto do notebook
def pipe(inputs):
    # Garante suporte a lotes (batches) ou itens individuais de forma transparente
    single_input = False
    if isinstance(inputs, str):
        inputs = [inputs]
        single_input = True
    elif hasattr(inputs, "__iter__") and not isinstance(inputs, list):
        # Se for um KeyDataset ou gerador, converte para lista
        inputs = [item for item in inputs]

    results = []
    for text in inputs:
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to("cuda:0")
        outputs = model.generate(input_ids, max_new_tokens=50)
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        results.append([{"generated_text": generated_text}])

    return results[0] if single_input else results

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [19]:
# Prepare our data
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example['text']})
data

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})

In [26]:
pd.DataFrame(data["test"])

,text,label,t5
0,lovingly photographed in the manner of a golde...,1,Is the following sentence positive or negative...
1,consistently clever and suspenseful .,1,Is the following sentence positive or negative...
2,"it's like a "" big chill "" reunion of the baade...",1,Is the following sentence positive or negative...
3,the story gives ample opportunity for large-sc...,1,Is the following sentence positive or negative...
4,"red dragon "" never cuts corners .",1,Is the following sentence positive or negative...
...,...,...,...
1061,a terrible movie that some people will neverth...,0,Is the following sentence positive or negative...
1062,there are many definitions of 'time waster' bu...,0,Is the following sentence positive or negative...
1063,"as it stands , crocodile hunter has the hurrie...",0,Is the following sentence positive or negative...
1064,the thing looks like a made-for-home-video qui...,0,Is the following sentence positive or negative...


In [20]:
# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)

100%|██████████| 1066/1066 [00:00<00:00, 1147914.78it/s]


In [21]:
evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.83      0.85      0.84       533
Positive Review       0.85      0.83      0.84       533

       accuracy                           0.84      1066
      macro avg       0.84      0.84      0.84      1066
   weighted avg       0.84      0.84      0.84      1066



# **Convertendo task específicas para instruções textuais**


<br>

![imagem](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/fig4-21.png?raw=true)

 $\color{blue}{\text{Você agora...}}$

 ### Como você faria um modelo de sumarização ou tradução empregando um modelo geral de LLM?